In [ ]:
!git clone https://github.com/VasiliBaranov/packing-generation.git

Cloning into 'packing-generation'...
remote: Enumerating objects: 3964, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 3964 (delta 48), reused 91 (delta 40), pack-reused 3857 (from 1)
Receiving objects: 100% (3964/3964), 5.94 MiB | 7.78 MiB/s, done.
Resolving deltas: 100% (1681/1681), done.


In [ ]:
%cd /content/packing-generation/_Release

/content/packing-generation/_Release


In [ ]:
%pwd

'/content/packing-generation/_Release'

In [ ]:
!make

Streaming output truncated to the last 5000 lines.
                 from ../PackingGeneration/Generation/PackingServices/Source/../Headers/ClosestJammingVelocityProvider.h:15,
                 from ../PackingGeneration/Generation/PackingServices/Source/ClosestJammingVelocityProvider.cpp:5:
../Externals/Eigen/Eigen/src/SparseCore/SparseMatrix.h: In function ‘void Eigen::internal::set_from_triplets(const InputIterator&, const InputIterator&, SparseMatrixType&, int)’:
../Externals/Eigen/Eigen/src/SparseCore/SparseMatrix.h:1024:44: warning: typedef ‘Index’ locally defined but not used []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-local-typedefs-Wunused-local-typedefs]8;;]
 1024 |   typedef typename SparseMatrixType::Index Index;
      |                                            ^~~~~
In file included from ../Externals/Eigen/Eigen/SparseCore:59,
                 from ../Externals/Eigen/Eigen/Sparse:17,
                 from ../PackingGeneration/Generation/P

In [ ]:
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def writetemplate(line, cell):
    with open(line, 'w') as f:
        f.write(cell.format(**globals()))

In [ ]:
from random import randint
num_particles = 718
#seed = randint(0, 1000)
seed = 809
print(seed)

809


In [ ]:
import numpy as np
from math import pi
X = np.repeat(6.0, num_particles)
np.savetxt("diameters.txt", X)
particles_vol = sum(4/3 * pi * (X/2)**3)
box_length = 50.0
print(box_length)

50.0


In [ ]:
%%writetemplate generation.conf
Particles count: {num_particles}
Packing size: {box_length} {box_length} {box_length}
Generation start: 1
Seed: {seed}
Steps to write: 1000
Boundaries mode: 1
Contraction rate: 1e-3

In [ ]:
%%time
!rm packing.xyzd packing_init.xyzd packing_prev.xyzd contraction_energies.txt packing.nfo
!./PackingGeneration.exe -fba

rm: cannot remove 'packing.xyzd': No such file or directory
rm: cannot remove 'packing_init.xyzd': No such file or directory
rm: cannot remove 'packing_prev.xyzd': No such file or directory
rm: cannot remove 'contraction_energies.txt': No such file or directory
rm: cannot remove 'packing.nfo': No such file or directory
LittleEndian
The current working directory is /content/packing-generation/_Release.

N is 718
dimensions are 50.000000 50.000000 50.000000
generation mode: start
seed 809
steps to write intermediate state 1000
boundaries mode is 1


Reading particle diameters from a file 'diameters'...
done.
Contraction rate is 0.001
Total volume of particles is 81203.9
Theoretical porosity is 0.350369
Global minimum is 0.0683905
Step 0. Inner diameter ratio is 0.103694331605996. Outer diameter ratio is 1.062127239898020. Writing int. packing state...Contraction rate: 1.000000e-03
done.
Checking...
Calc. porosity is 0.393073
Theoretical porosity is 0.350369
Checking min particle distance

In [ ]:
%%writetemplate generation.conf
Particles count: {num_particles}
Packing size: {box_length} {box_length} {box_length}
Generation start: 0
Seed: {seed}
Steps to write: 1000
Boundaries mode: 1
Contraction rate: 1e-3

In [ ]:
!rm packing.nfo
!./PackingGeneration.exe -ls

In [ ]:
!rm packing.nfo
!./PackingGeneration.exe -lsgd

In [ ]:
"""https://github.com/VasiliBaranov/packing-generation/issues/30#issue-1103925864"""
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

packing = np.fromfile('packing.xyzd').reshape(-1, 4)

with open('packing.nfo', "r+") as nfo:
    lines = nfo.readlines()
    Theoretical_Porosity = float(lines[2].split()[2])
    Final_Porosity = float(lines[3].split()[2])
    # print(Theoretical_Porosity, Final_Porosity)

    scaling_factor = ((1 - Final_Porosity) / (1 - Theoretical_Porosity)) ** (1 / 3)


    real_diameters = packing[:, 3] * scaling_factor
    actual_density = sum((4/3) * pi * (np.array(real_diameters) / 2)**3) / box_length**3
    print(actual_density)
    packing[:, 3] = real_diameters
    packing.tofile('packing.xyzd')        # updating the packing: this line will modifies diameters in the packing.xyzd

    # update packing.nfo and set TheoreticalPorosity to FinalPorosity to avoid scaling the packing once again the next time running this script.
    lines[3] = lines[3].replace(str(Final_Porosity), str(Theoretical_Porosity))
    nfo.seek(0)
    nfo.writelines(lines)


def ms(x, y, z, radius, resolution=10):
    """Return the coordinates for plotting a sphere centered at (x,y,z)"""
    u, v = np.mgrid[0:2*np.pi:resolution*2j, 0:np.pi:resolution*1j]
    X = radius * np.cos(u)*np.sin(v) + x
    Y = radius * np.sin(u)*np.sin(v) + y
    Z = radius * np.cos(v) + z
    return (X, Y, Z)

df = pd.DataFrame(packing, columns=["x", "y", "z", "d"])
df["r"] = df["d"] / 2
print(df)

# data = []
# for index, row in df.iterrows():
#     (x_pns_surface, y_pns_surface, z_pns_suraface) = ms(row.x, row.y, row.z, row.r)
#     data.append(go.Surface(x=x_pns_surface, y=y_pns_surface, z=z_pns_suraface, opacity=0.95))

# fig = go.Figure(data=data)
# fig.show()

In [ ]:
from scipy.spatial.distance import pdist, squareform
xyz = df[["x", "y", "z"]].values
dm = pdist(xyz)
min(dm / 2) - df.iloc[0]["r"]

In [ ]:
import plotly.express as px
px.histogram(dm)

In [ ]:
from sklearn.neighbors import NearestNeighbors
neigh = NearestNeighbors(n_neighbors=5, radius=box_length)
neigh.fit(xyz)
neigh_dist, neigh_ind = neigh.kneighbors(xyz)

In [ ]:
neigh_dist[:,1]